# 🤝 Cookbook 5: Multi-Agent Systems

Learn how to build systems where multiple agents work together!

**Topics covered:**
1. Agents as tools
2. Agent coordination patterns
3. Building agent workflows
4. Real-world multi-agent example

**Prerequisites**: Complete Cookbooks 1-4 first!

---

In [ ]:
!pip install strands-agents strands-agents-tools -q
print("✅ Ready to go!")

## 🎯 Why Multi-Agent Systems?

Complex problems often benefit from specialized agents:

| Pattern | Description | Use Case |
|---------|-------------|----------|
| **Agents as Tools** | One agent uses another as a tool | Specialization |
| **Pipeline** | Agents process data sequentially | Data processing |
| **Coordinator** | One agent orchestrates others | Complex tasks |
| **Debate** | Agents critique each other | Quality assurance |

## 🔧 Step 1: Agents as Tools

The simplest way to use multiple agents: wrap one as a tool for another!

In [ ]:
from strands import Agent, tool
from strands_tools import calculator

# Create a specialized math expert agent
math_expert = Agent(
    system_prompt="""You are a math expert. 
    Explain mathematical concepts clearly and solve problems step-by-step.
    Keep explanations concise.""",
    tools=[calculator]
)

# Wrap the math expert as a tool
@tool
def ask_math_expert(question: str) -> str:
    """Ask the math expert agent a question.
    
    Args:
        question: A math-related question to ask the expert.
    
    Returns:
        The expert's response.
    """
    result = math_expert(question)
    return str(result)

# Create a main agent that can use the math expert
main_agent = Agent(
    system_prompt="""You are a helpful assistant. 
    When you encounter math questions, use the math expert tool.
    For other questions, answer directly.""",
    tools=[ask_math_expert]
)

# Test it!
response = main_agent("Can you explain what a prime number is and give me the first 5 prime numbers?")

print("\n" + "="*50)
print("✅ The main agent delegated to the math expert!")

## 👥 Step 2: Creating Specialized Agents

Let's create multiple specialized agents for different domains.

In [ ]:
from strands import Agent, tool
from strands_tools import calculator

# Create specialized agents
code_expert = Agent(
    system_prompt="""You are a Python programming expert.
    Write clean, well-commented code.
    Explain your code briefly."""
)

writing_expert = Agent(
    system_prompt="""You are a professional writer and editor.
    Help with writing, grammar, and style.
    Be concise but thorough."""
)

math_expert = Agent(
    system_prompt="""You are a mathematics expert.
    Solve problems step-by-step.
    Explain concepts clearly.""",
    tools=[calculator]
)

# Wrap each as a tool
@tool
def ask_coder(question: str) -> str:
    """Ask the coding expert for help with programming.
    
    Args:
        question: A programming-related question.
    """
    return str(code_expert(question))

@tool
def ask_writer(text: str) -> str:
    """Ask the writing expert for help with text.
    
    Args:
        text: Text to review or a writing request.
    """
    return str(writing_expert(text))

@tool
def ask_mathematician(question: str) -> str:
    """Ask the math expert for help with mathematics.
    
    Args:
        question: A math-related question.
    """
    return str(math_expert(question))

# Create a coordinator agent
coordinator = Agent(
    system_prompt="""You are a helpful coordinator assistant.
    You have access to three expert agents:
    - Coding expert: For programming questions
    - Writing expert: For writing and editing
    - Math expert: For mathematics
    
    Route questions to the appropriate expert(s) and synthesize their responses.""",
    tools=[ask_coder, ask_writer, ask_mathematician]
)

# Test the coordinator
response = coordinator("""
I need help with two things:
1. Write a short Python function to check if a number is prime
2. Review this sentence for grammar: "The cats is sleeping on the couch."
""")

print("\n" + "="*50)
print("✅ Coordinator delegated to multiple experts!")

## 🔄 Step 3: Agent Pipeline

Chain agents together where output of one becomes input to the next.

In [ ]:
from strands import Agent

def create_pipeline():
    """Create a content creation pipeline."""
    
    # Stage 1: Research/brainstorm
    researcher = Agent(
        system_prompt="""You are a research assistant.
        Given a topic, provide 3-5 key points that would be interesting to cover.
        Be factual and informative."""
    )
    
    # Stage 2: Write content
    writer = Agent(
        system_prompt="""You are a content writer.
        Given a topic and key points, write a short, engaging article (2-3 paragraphs).
        Make it accessible and interesting."""
    )
    
    # Stage 3: Edit and polish
    editor = Agent(
        system_prompt="""You are an editor.
        Given an article, improve it by:
        - Fixing any grammar issues
        - Improving clarity
        - Adding a compelling title
        Return the polished version."""
    )
    
    return researcher, writer, editor

def run_pipeline(topic: str):
    """Run the content pipeline on a topic."""
    researcher, writer, editor = create_pipeline()
    
    print(f"📚 Stage 1: Researching '{topic}'...")
    research = researcher(f"Research the topic: {topic}")
    print(f"\n✅ Research complete!")
    
    print(f"\n✍️ Stage 2: Writing content...")
    draft = writer(f"Write an article about: {topic}\n\nKey points from research: {research}")
    print(f"\n✅ Draft complete!")
    
    print(f"\n🔍 Stage 3: Editing...")
    final = editor(f"Edit and polish this article:\n\n{draft}")
    print(f"\n✅ Final article ready!")
    
    return final

# Run the pipeline
result = run_pipeline("the benefits of learning to code")

print("\n" + "="*50)
print("✅ Content pipeline completed!")

## 💭 Step 4: Debate Pattern

Two agents debate to improve the quality of responses.

In [ ]:
from strands import Agent

def create_debate_system():
    """Create a debate system with advocate and critic."""
    
    advocate = Agent(
        system_prompt="""You are an advocate who proposes solutions.
        Present your ideas clearly and defend them.
        Be concise but thorough."""
    )
    
    critic = Agent(
        system_prompt="""You are a constructive critic.
        Analyze proposals for potential issues and suggest improvements.
        Be fair but thorough in your critique."""
    )
    
    synthesizer = Agent(
        system_prompt="""You are a synthesizer.
        Given a proposal and critique, create an improved final solution.
        Combine the best of both perspectives."""
    )
    
    return advocate, critic, synthesizer

def debate_and_synthesize(problem: str):
    """Run a debate to solve a problem."""
    advocate, critic, synthesizer = create_debate_system()
    
    print(f"🎯 Problem: {problem}")
    print("\n" + "="*50)
    
    # Round 1: Advocate proposes
    print("\n🗣️ ADVOCATE:")
    proposal = advocate(f"Propose a solution for: {problem}")
    
    # Round 2: Critic reviews
    print("\n\n🔍 CRITIC:")
    critique = critic(f"Critique this proposal:\n\n{proposal}")
    
    # Round 3: Synthesize
    print("\n\n⚖️ SYNTHESIZER:")
    final = synthesizer(f"""
    Original proposal:
    {proposal}
    
    Critique:
    {critique}
    
    Create an improved final solution.
    """)
    
    return final

# Run the debate
result = debate_and_synthesize("How should a beginner start learning programming?")

print("\n" + "="*50)
print("✅ Debate completed with synthesized solution!")

## 🏢 Step 5: Real-World Example: Customer Support System

Build a multi-agent customer support system!

In [ ]:
from strands import Agent, tool

# Knowledge base (simulated)
KNOWLEDGE_BASE = {
    "refund": "Refunds are processed within 5-7 business days. You can request a refund within 30 days of purchase.",
    "shipping": "Standard shipping takes 3-5 days. Express shipping takes 1-2 days for an additional $10.",
    "returns": "Items can be returned within 30 days in original condition. Free returns for defective items.",
    "account": "You can reset your password from the login page. Contact support if you need additional help.",
}

@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base for information.
    
    Args:
        query: The topic to search for (refund, shipping, returns, account).
    """
    query_lower = query.lower()
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower:
            return value
    return "No information found. Please contact human support."

@tool
def create_ticket(issue: str, priority: str = "normal") -> dict:
    """Create a support ticket for human review.
    
    Args:
        issue: Description of the issue.
        priority: Priority level (low, normal, high).
    """
    import random
    ticket_id = random.randint(10000, 99999)
    return {
        "ticket_id": ticket_id,
        "status": "created",
        "priority": priority,
        "message": f"Ticket #{ticket_id} created. A support agent will contact you within 24 hours."
    }

# Tier 1: Initial support agent
tier1_agent = Agent(
    system_prompt="""You are a friendly Tier 1 customer support agent.
    Your job is to:
    1. Greet the customer warmly
    2. Understand their issue
    3. Search the knowledge base for answers
    4. Provide helpful responses
    
    If you can't resolve the issue, escalate by creating a ticket.""",
    tools=[search_knowledge_base, create_ticket]
)

# Supervisor agent
@tool
def escalate_to_tier1(customer_message: str) -> str:
    """Send customer message to Tier 1 support.
    
    Args:
        customer_message: The customer's message or issue.
    """
    return str(tier1_agent(customer_message))

supervisor = Agent(
    system_prompt="""You are a customer support supervisor.
    Route incoming requests to Tier 1 support.
    Monitor responses for quality.
    Add helpful context when needed.""",
    tools=[escalate_to_tier1]
)

# Test the system
print("Customer: I want to return an item I bought last week. How do I do that?")
print("\n" + "="*50 + "\n")

response = supervisor("Customer inquiry: I want to return an item I bought last week. How do I do that?")

print("\n" + "="*50)
print("✅ Multi-agent support system in action!")

## 🏋️ Exercise: Build Your Own Multi-Agent System

Design a multi-agent system for a specific use case!

In [ ]:
from strands import Agent, tool

# 🏋️ YOUR EXERCISE: Build a multi-agent system!
#
# Ideas:
# - A content moderation system (detector + reviewer + decision maker)
# - A code review system (analyzer + security checker + summary writer)
# - A travel planning system (researcher + budget calculator + itinerary maker)
# - A study assistant (topic explainer + quiz maker + answer checker)

# Template:
# agent1 = Agent(system_prompt="...", tools=[...])
# agent2 = Agent(system_prompt="...", tools=[...])

# @tool
# def ask_agent1(query: str) -> str:
#     """Ask agent 1."""
#     return str(agent1(query))

# coordinator = Agent(system_prompt="...", tools=[ask_agent1, ...])

print("💡 Design your own multi-agent system!")
print("\nConsider:")
print("- What specialized agents do you need?")
print("- How should they communicate?")
print("- What tools does each agent need?")

## ✅ Summary

You've learned:

1. **Agents as Tools** - Wrap agents to use them as tools
2. **Specialized Agents** - Create domain experts
3. **Coordinator Pattern** - One agent orchestrates others
4. **Pipeline Pattern** - Chain agents sequentially
5. **Debate Pattern** - Agents critique and improve each other
6. **Real-world Systems** - Customer support example

---

## 🎓 Course Complete!

Congratulations! You've completed all 5 cookbooks and learned:

1. ✅ **Getting Started** - Basic agent creation
2. ✅ **Custom Tools** - Building your own tools
3. ✅ **Model Providers** - Using different AI models
4. ✅ **Conversations** - Managing memory and context
5. ✅ **Multi-Agent Systems** - Coordinating multiple agents

---

## 📚 Resources

- [Multi-Agent Patterns](https://strandsagents.com/latest/user-guide/concepts/multi-agent/multi-agent-patterns/)
- [Agents as Tools](https://strandsagents.com/latest/user-guide/concepts/multi-agent/agents-as-tools/)
- [Strands Samples](https://github.com/strands-agents/samples)